Problems: need to strip the start of name (has professor and stuff)
some profs dont exist? but ill still include (its Dr Chris Bell)
Some people have the title to not actually be their prof level, instead its in their name



Stephen Gray excluded — both UQ profile pages (business.uq.edu.au/profile/830 
and about.uq.edu.au/experts/302) went 404 mid-session. eSpace author id is 533, 
70 records. Add manually if needed.


In [6]:
#pip install requests beautifulsoup4 pandas openpyxl ipykernel


In [ ]:
import requests, time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

import re

PREFIX = re.compile(
    r"^(Associate Professor|Emeritus Professor|Professor|Dr|Mr|Mrs|Ms|Miss|A/Prof|Prof|Assoc\.? Prof\.?)\.?\s+",
    re.IGNORECASE
)


LADDER = [
    ("Emeritus Professor",  r"emeritus prof"),
    ("Associate Professor", r"associate prof|a/prof"),
    ("Associate Lecturer",  r"associate lecturer"),
    ("Senior Lecturer",     r"senior lecturer"),
    ("Senior Research Fellow", r"senior research fellow"),
    ("Research Fellow",     r"research fellow"),
    ("Teaching Associate",  r"teaching associate"),
    ("Professor",           r"\bprofessor\b|chair in"),
    ("Lecturer",            r"\blecturer\b"),
]

LEVEL = {
    "Associate Lecturer":      "A",
    "Lecturer":                "B",
    "Fellow":                  "B",
    "Research Fellow":         "B",
    "Senior Lecturer":         "C",
    "Senior Fellow":           "C",
    "Senior Research Fellow":  "C",
    "Associate Professor":     "D",
    "Professor":               "E",
    "Professorial Fellow":     "E",
    "Professor Emeritus":      "E",
    "Emeritus Professor":      "E",
}

def rank(title, prefix):
    for label, pat in LADDER:
        if title and re.search(pat, title, re.I):
            return label
    if prefix and prefix.lower() not in {"dr", "mr", "mrs", "ms", "miss"}:
        return prefix
    return None


def alive(url):
    try:
        return requests.head(url, allow_redirects=True, timeout=10).status_code == 200
    except requests.RequestException:
        return False


# loop 1 — extraction only, no network

TARGETS = [
    ("https://business.uq.edu.au/team/finance-discipline", "University of Queensland", "Finance"),
    ("https://business.uq.edu.au/team/accounting-discipline", "University of Queensland", "Accounting")
]


# TARGETS = [
#     ("https://business.uq.edu.au/team/finance-discipline", "University of Queensland", "Finance")
# ]

records = []
for url, uni, disc in TARGETS:
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    cards = soup.select(".person--teaser")
    print(f"{disc}: {len(cards)} cards")
    for card in cards:
        link = card.select_one(".person__display-name a")
        if not link:
            continue
        name = link.get_text(strip=True)
        m = PREFIX.match(name)
        href = link["href"]

        titles = [t.get_text(strip=True) for t in card.select(".position__title")]
        titles = [t for t in titles if t]
        substantive = [t for t in titles if not t.startswith("Affiliate")]
        title = (substantive or titles or [None])[0]

        records.append({
            "university": uni,
            "discipline": disc,
            "name": name,
            "name_clean": PREFIX.sub("", name).strip(),
            "prefix": m.group(1) if m else None,
            "title": title,
            "title_clean": rank(title, m.group(1) if m else None),
            "profile_url": urljoin(url, href),
        })

    print(len(records))  # expect 24

# loop 2 — network checks
for r in records:
    r["alive"] = alive(r["profile_url"])
    time.sleep(1)

Finance: 24 cards
24
Accounting: 19 cards
43


In [8]:
for r in records:
    print(f"{r['title_clean']!s:25} <- {r['title']}")

Associate Lecturer        <- Associate Lecturer
None                      <- None
Professor                 <- Professor
Associate Lecturer        <- Associate Lecturer (Finance)
Teaching Associate        <- Teaching Associate
Senior Lecturer           <- Senior Lecturer
Senior Lecturer           <- Senior Lecturer
Professor                 <- Malcolm Broomhead Chair in Finance & Program Convenor (Bachelor of Advanced Finance and Economics) of UQ Business School
Senior Lecturer           <- Senior Lecturer in Finance
Lecturer                  <- Lecturer in Finance
Senior Lecturer           <- Senior Lecturer
Lecturer                  <- Lecturer
Lecturer                  <- Lecturer in Finance
Associate Professor       <- Discipline Convenor, Finance
Senior Lecturer           <- Senior Lecturer in Finance
Senior Lecturer           <- Senior Lecturer in Finance
Associate Professor       <- Associate Professor & Program Convenor (Bachelor of Commerce) of UQ Business School & Program Con

In [9]:
KEEP = ["university", "discipline", "name_clean", "title_clean", "profile_url"]
records = [{k: r[k] for k in KEEP} for r in records]
records

[{'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Jon Aster',
  'title_clean': 'Associate Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/9542/jon-aster'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Chris Bell',
  'title_clean': None,
  'profile_url': 'https://business.uq.edu.au/profile/17730/chris-bell'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Shaun Bond',
  'title_clean': 'Professor',
  'profile_url': 'https://business.uq.edu.au/profile/6239/shaun-bond'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Alexander Cameron',
  'title_clean': 'Associate Lecturer',
  'profile_url': 'https://business.uq.edu.au/profile/18606/alexander-cameron'},
 {'university': 'University of Queensland',
  'discipline': 'Finance',
  'name_clean': 'Yong Ming Chen',
  'title_clean': 'Teaching Associate',
  'profile_url': 'h

In [10]:
#I dont know what Stephen Gray

In [11]:
API_URL = "https://api.library.uq.edu.au/v1/records/search?export_to=&page=1&per_page=100&sort=published_date&order_by=desc&mode=advanced&key%5Brek_author_id%5D=533"

HEADERS = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "en-US,en-AU;q=0.9,en;q=0.8",
    "origin": "https://espace.library.uq.edu.au",
    "referer": "https://espace.library.uq.edu.au/",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-site",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36",
}

r = requests.get(API_URL, headers=HEADERS, timeout=10)
print(r.status_code, r.text[:300])

200 {"total":70,"took":33,"per_page":100,"current_page":1,"from":1,"to":70,"data":[{"rek_pid":"UQ:092048b","rek_title_xsdmf_id":null,"rek_title":"Sampling error and the joint estimation of imputation credit value and cash dividend value","rek_description_xsdmf_id":null,"rek_description":"The value of im


In [12]:
import json
d = r.json()
print(d["total"], d["per_page"], d["current_page"])
print(json.dumps(d["data"][0], indent=2))

70 100 1
{
  "rek_pid": "UQ:092048b",
  "rek_title_xsdmf_id": null,
  "rek_title": "Sampling error and the joint estimation of imputation credit value and cash dividend value",
  "rek_description_xsdmf_id": null,
  "rek_description": "The value of imputation credits can only be estimated jointly with the value of cash dividends. We show that random variation across samples leads to estimates of credit value that move in the opposite direction to estimates of cash value. Derivative prices suggest a value for credits of 0.01 to 0.20 (0.01 to 0.07 if cash is worth 0.94, and 0.13 to 0.20 if cash is worth 0.87). Ex-dividend prices suggest a value for credits of 0.23 to 0.46 (0.23 to 0.36 if cash is worth 0.85, and 0.33 to 0.46 if cash is worth 0.75).",
  "rek_display_type_xsdmf_id": null,
  "rek_display_type": 179,
  "rek_status_xsdmf_id": null,
  "rek_status": 2,
  "rek_date_xsdmf_id": null,
  "rek_date": "2023-04-01T00:00:00Z",
  "rek_object_type_xsdmf_id": null,
  "rek_object_type": 3,
 

In [13]:
rec = r.json()["data"][0]
print([k for k in rec if "journal" in k.lower()])

['fez_record_search_key_journal_name', 'fez_record_search_key_language_of_journal_name', 'fez_record_search_key_native_script_journal_name', 'fez_record_search_key_roman_script_journal_name', 'fez_record_search_key_translated_journal_name', 'fez_matched_journals']


In [14]:
print(rec["fez_record_search_key_journal_name"])

{'rek_journal_name_id': 5458025, 'rek_journal_name_pid': 'UQ:092048b', 'rek_journal_name_xsdmf_id': None, 'rek_journal_name': 'Accounting and Finance'}


In [15]:
for p in records:
    if p.get("espace_id"):
        continue
    resp = requests.get(p["profile_url"], headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    link = BeautifulSoup(resp.text, "html.parser").select_one('a[href*="author_id"]')
    p["espace_id"] = link["href"].rstrip("/").split("/")[-1] if link else None
    print(p["name_clean"], resp.status_code, p["espace_id"])
    time.sleep(1)

print(sum(1 for p in records if p["espace_id"]), "of", len(records))

Jon Aster 200 None
Chris Bell 404 None
Shaun Bond 200 6079108
Alexander Cameron 200 97219
Yong Ming Chen 200 7627091
Hasibul Chowdhury 200 2254822
Nicolas Eugster 200 7275857
Stephen Gray 404 None
Khoa Hoang 200 87704
Weiting Hu 200 7639113
Ronghong Huang 200 87707
Shirina Lin 200 6004495
Leo Luong 200 5497249
Jacquelyn Humphrey 200 6336
Lin Mi 200 78774
Suman Neupane-Joshi 200 4329687
Lily Nguyen 200 5508774
Vanitha Ragunathan 200 1055
Dewan Rahman 200 2284981
Saphira Rekker 200 93285
Eric Tan 200 4818628
Kelvin Tan 200 8472
Elizabeth Zhu 200 92037
Min Zhu 200 4503669
Ankit Jain 200 4930238
Byungki Kim 200 4850515
Kirsty Dunbar 200 7635606
Bobae Choi 200 7646522
Debbie Jeffery 200 None
Mia Han 200 7642804
Kathleen Herbohn 200 1736
Grace Hsu 200 1009
Mark Wallis 200 4940317
Matthew Peters 200 3804387
Michael Turner 200 77962
Natalie Peng 200 87748
Peter Clarkson 200 326
Peter Do 200 4469982
Robyn King 200 6621
Sergeja Slapnicar 200 5402627
Xin Yu 200 115914
Yang Xu 200 4336037
Muhammad

In [16]:
[p["name_clean"] for p in records if not p["espace_id"]]

['Jon Aster', 'Chris Bell', 'Stephen Gray', 'Debbie Jeffery']

In [17]:
print(len(records), sum(1 for p in records if p["espace_id"]))

43 39


In [ ]:
import requests, time

orcids = {}
for p in records:
    if not p.get("espace_id"):
        continue
    url = (f"{ESPACE_BASE}?export_to=&page=1&per_page=5&sort=published_date"
           f"&order_by=desc&mode=advanced&key%5Brek_author_id%5D={p['espace_id']}")
    try:
        d = requests.get(url, headers=HEADERS, timeout=20).json()
    except requests.RequestException:
        continue
    found = None
    for rec in d.get("data", []):
        for a in (rec.get("fez_record_search_key_author_id") or []):
            au = a.get("author") or {}
            if str(a.get("rek_author_id")) == str(p["espace_id"]) and au.get("aut_orcid_id"):
                found = au["aut_orcid_id"]
                break
        if found:
            break
    orcids[p["name_clean"]] = found
    time.sleep(0.5)

for p in records:
    p["orcid"] = orcids.get(p["name_clean"])

print(sum(1 for p in records if p.get("orcid")), "of", len(records), "have an ORCID")

In [ ]:
ESPACE_BASE  = "https://api.library.uq.edu.au/v1/records/search"

pubs = []
for i, p in enumerate(records, 1):
    if not p.get("espace_id"):
        print(f"{i} {p['name_clean']}: no eSpace ID, skipped")
        continue

    page, fetched, total = 1, 0, None
    while True:
        url = (f"{ESPACE_BASE}?export_to=&page={page}&per_page=100&sort=published_date"
               f"&order_by=desc&mode=advanced&key%5Brek_author_id%5D={p['espace_id']}")
        d = requests.get(url, headers=HEADERS, timeout=15).json()
        total = d["total"]

        for rec in d["data"]:
            jn = rec.get("fez_record_search_key_journal_name")

            mj = rec.get("fez_matched_journals") or []
            mj = mj[0] if isinstance(mj, list) and mj else (mj if isinstance(mj, dict) else None)
            fj = (mj or {}).get("fez_journal") or {}

            issns = [s.get("rek_issn") for s in (rec.get("fez_record_search_key_issn") or []) if s.get("rek_issn")]

            auths = rec.get("fez_record_search_key_author") or []
            aids  = rec.get("fez_record_search_key_author_id") or []
            doi = rec.get("fez_record_search_key_doi")


            pubs.append({
                "name": p["name_clean"],
                "espace_id": p["espace_id"],
                "title": rec.get("rek_title"),
                "year": rec["rek_date"][:4] if rec.get("rek_date") else None,
                "type": rec.get("rek_genre"),
                "n_authors": len(auths),
                "authors": "; ".join(a.get("rek_author", "") for a in
                                     sorted(auths, key=lambda a: a.get("rek_author_order", 0))),
                "author_ids": [a.get("rek_author_id") for a in aids if a.get("rek_author_id")],
                "issns": issns,
                "journal": jn["rek_journal_name"] if jn else None,
                "journal_id": fj.get("jnl_jid"),
                "journal_canonical": fj.get("jnl_title"),
                "publisher": fj.get("jnl_publisher"),
                "doi": doi.get("rek_doi") if isinstance(doi, dict) else None,

                "link": f"https://espace.library.uq.edu.au/view/{rec['rek_pid']}",
            })

        fetched += len(d["data"])
        if fetched >= total or not d["data"]:
            break
        page += 1
        time.sleep(1)

    flag = "" if fetched == total else "  <-- MISMATCH"
    print(f"{i} {p['name_clean']}: {total} total, {fetched} fetched{flag}")
    time.sleep(1)

print(len(pubs), "records from", len({x['name'] for x in pubs}), "people")

1 Jon Aster: no eSpace ID, skipped
2 Chris Bell: no eSpace ID, skipped
3 Shaun Bond: 36 total, 36 fetched
4 Alexander Cameron: 0 total, 0 fetched
5 Yong Ming Chen: 2 total, 2 fetched
6 Hasibul Chowdhury: 33 total, 33 fetched
7 Nicolas Eugster: 12 total, 12 fetched
8 Stephen Gray: no eSpace ID, skipped
9 Khoa Hoang: 12 total, 12 fetched
10 Weiting Hu: 2 total, 2 fetched
11 Ronghong Huang: 9 total, 9 fetched
12 Shirina Lin: 1 total, 1 fetched
13 Leo Luong: 10 total, 10 fetched
14 Jacquelyn Humphrey: 37 total, 37 fetched
15 Lin Mi: 15 total, 15 fetched
16 Suman Neupane-Joshi: 28 total, 28 fetched
17 Lily Nguyen: 22 total, 22 fetched
18 Vanitha Ragunathan: 20 total, 20 fetched
19 Dewan Rahman: 16 total, 16 fetched
20 Saphira Rekker: 41 total, 41 fetched
21 Eric Tan: 16 total, 16 fetched
22 Kelvin Tan: 31 total, 31 fetched
23 Elizabeth Zhu: 29 total, 29 fetched
24 Min Zhu: 36 total, 36 fetched
25 Ankit Jain: 9 total, 9 fetched
26 Byungki Kim: 3 total, 3 fetched
27 Kirsty Dunbar: 3 total, 3 

In [19]:
from collections import Counter
print(Counter(x["type"] for x in pubs))
print(sum(1 for x in pubs if not x["journal"]), "with no journal")
print(sum(1 for x in pubs if not x["year"]), "with no year")
print(Counter(x["name"] for x in pubs).most_common())

Counter({'Journal Article': 664, 'Conference Paper': 80, 'Research Report': 29, 'Book Chapter': 28, 'Newspaper Article': 22, 'Thesis': 15, 'Working Paper': 13, 'Audio Document': 12, 'Data Collection': 9, 'Book': 8, 'Creative Work': 2, 'Generic Document': 2, 'Conference Proceedings': 1, 'Department Technical Report': 1, 'Video Document': 1})
217 with no journal
0 with no year
[('Sergeja Slapnicar', 86), ('Peter Clarkson', 62), ('Kathleen Herbohn', 61), ('Muhammad Nadeem', 50), ('Saphira Rekker', 41), ('Jacquelyn Humphrey', 37), ('Michael Turner', 37), ('Shaun Bond', 36), ('Min Zhu', 36), ('Hasibul Chowdhury', 33), ('Natalie Peng', 32), ('Kelvin Tan', 31), ('Elizabeth Zhu', 29), ('Xin Yu', 29), ('Suman Neupane-Joshi', 28), ('Bobae Choi', 27), ('Lily Nguyen', 22), ('Robyn King', 21), ('Vanitha Ragunathan', 20), ('Mark Wallis', 20), ('Dewan Rahman', 16), ('Eric Tan', 16), ('Matthew Peters', 16), ('Lin Mi', 15), ('Nicolas Eugster', 12), ('Khoa Hoang', 12), ('Leo Luong', 10), ('Ronghong Huan

In [20]:
print(sum(1 for x in pubs if x["type"] == "Journal Article" and not x["journal"]))

0


In [21]:
from collections import Counter
print(Counter(x["type"] for x in pubs if not x["journal"]))

Counter({'Conference Paper': 74, 'Research Report': 29, 'Book Chapter': 28, 'Newspaper Article': 22, 'Thesis': 15, 'Working Paper': 13, 'Audio Document': 12, 'Data Collection': 9, 'Book': 8, 'Creative Work': 2, 'Generic Document': 2, 'Conference Proceedings': 1, 'Department Technical Report': 1, 'Video Document': 1})


In [22]:
no_journal = [x for x in pubs if not x["journal"]]
print(len(no_journal))

import json
print(json.dumps(no_journal[0], indent=2))

217
{
  "name": "Shaun Bond",
  "espace_id": "6079108",
  "title": "Building business resilience",
  "year": "2022",
  "type": "Research Report",
  "n_authors": 16,
  "authors": "Hiller, Michael; Jenkinson, Lisa; Kastelle, Tim; Barrett, Andrew; Bond, Shaun; Miller, Andy; Bongiovanni, Ivano; Wilson, Andrew; Brown, Dan; Hubbard, Ellie; McLennan, Kelly; Schleicher, Sabine; Clark, Samantha; Wang, Jie; Gabby Walters; Meath, Cristyn",
  "author_ids": [
    6517,
    6079108,
    6736615,
    79393,
    87322,
    95158
  ],
  "issns": [],
  "journal": null,
  "journal_id": null,
  "journal_canonical": null,
  "publisher": null,
  "doi": null,
  "link": "https://espace.library.uq.edu.au/view/UQ:e5f0850"
}


In [23]:
pubs

[{'name': 'Shaun Bond',
  'espace_id': '6079108',
  'title': 'Mandatory disclosures of greenhouse gas emissions and managerial myopia: evidence from the United States',
  'year': '2025',
  'type': 'Journal Article',
  'n_authors': 4,
  'authors': 'Li, Chenjia; Chang, Liang; Bond, Shaun; Zhu, Yushu',
  'author_ids': [7635579, 6079108, 92037],
  'issns': ['1094-4060', '2213-3933'],
  'journal': 'The International Journal of Accounting',
  'journal_id': 11797,
  'journal_canonical': 'The International Journal of Accounting',
  'publisher': 'World Scientific Publishing Co. Pte. Ltd.',
  'doi': '10.1142/s1094406025410057',
  'link': 'https://espace.library.uq.edu.au/view/UQ:02e94da'},
 {'name': 'Shaun Bond',
  'espace_id': '6079108',
  'title': 'Large language models and financial market sentiment',
  'year': '2023',
  'type': 'Journal Article',
  'n_authors': 3,
  'authors': 'Bond, Shaun Alexander; Klok, Hayden; Zhu, Min',
  'author_ids': [6079108],
  'issns': ['1556-5068'],
  'journal': '

In [24]:
print(rec.get("fez_matched_journals"))

{'mtj_pid': 'UQ:114d942', 'mtj_jnl_id': 11920, 'mtj_status': 'M', 'fez_journal': {'jnl_jid': 11920, 'jnl_title': 'Measuring Business Excellence', 'jnl_abbrev_title': None, 'jnl_publisher': 'Emerald Publishing Limited', 'jnl_start_year': None, 'jnl_frequency': None, 'jnl_description': None, 'jnl_formats': None, 'jnl_is_refereed': None, 'jnl_source_id_era': '19372', 'jnl_advisory_statement': None, 'jnl_editing_user': None, 'jnl_editing_start_date': None, 'jnl_changeable_by_external_sources': True, 'jnl_created_date': '2021-02-22 01:33:14', 'jnl_updated_date': '2021-02-22 16:48:11', 'jnl_advisory_statement_type': None, 'is_diamond': None, 'fez_journal_era': [{'jnl_era_id': 103110, 'jnl_era_source_id': '19372', 'jnl_era_title': 'Measuring Business Excellence', 'jnl_era_source_year': 2023}, {'jnl_era_id': 11920, 'jnl_era_source_id': '19372', 'jnl_era_title': 'Measuring Business Excellence', 'jnl_era_source_year': 2018}, {'jnl_era_id': 32564, 'jnl_era_source_id': '19372', 'jnl_era_title': 'M

In [25]:
print(len(pubs), "rows,", len({x["name"] for x in pubs}), "people")

887 rows, 38 people


In [26]:
from collections import Counter
print(Counter(x["name"] for x in pubs).most_common())

[('Sergeja Slapnicar', 86), ('Peter Clarkson', 62), ('Kathleen Herbohn', 61), ('Muhammad Nadeem', 50), ('Saphira Rekker', 41), ('Jacquelyn Humphrey', 37), ('Michael Turner', 37), ('Shaun Bond', 36), ('Min Zhu', 36), ('Hasibul Chowdhury', 33), ('Natalie Peng', 32), ('Kelvin Tan', 31), ('Elizabeth Zhu', 29), ('Xin Yu', 29), ('Suman Neupane-Joshi', 28), ('Bobae Choi', 27), ('Lily Nguyen', 22), ('Robyn King', 21), ('Vanitha Ragunathan', 20), ('Mark Wallis', 20), ('Dewan Rahman', 16), ('Eric Tan', 16), ('Matthew Peters', 16), ('Lin Mi', 15), ('Nicolas Eugster', 12), ('Khoa Hoang', 12), ('Leo Luong', 10), ('Ronghong Huang', 9), ('Ankit Jain', 9), ('Grace Hsu', 9), ('Peter Do', 6), ('Mia Han', 5), ('Byungki Kim', 3), ('Kirsty Dunbar', 3), ('Yang Xu', 3), ('Yong Ming Chen', 2), ('Weiting Hu', 2), ('Shirina Lin', 1)]


In [27]:
from collections import Counter
c = Counter(x["name"] for x in pubs)
for p in records:
    if p.get("espace_id"):
        print(f"{p['name_clean']:22} {p['espace_id']:>9}  {c.get(p['name_clean'], 0)}")

Shaun Bond               6079108  36
Alexander Cameron          97219  0
Yong Ming Chen           7627091  2
Hasibul Chowdhury        2254822  33
Nicolas Eugster          7275857  12
Khoa Hoang                 87704  12
Weiting Hu               7639113  2
Ronghong Huang             87707  9
Shirina Lin              6004495  1
Leo Luong                5497249  10
Jacquelyn Humphrey          6336  37
Lin Mi                     78774  15
Suman Neupane-Joshi      4329687  28
Lily Nguyen              5508774  22
Vanitha Ragunathan          1055  20
Dewan Rahman             2284981  16
Saphira Rekker             93285  41
Eric Tan                 4818628  16
Kelvin Tan                  8472  31
Elizabeth Zhu              92037  29
Min Zhu                  4503669  36
Ankit Jain               4930238  9
Byungki Kim              4850515  3
Kirsty Dunbar            7635606  3
Bobae Choi               7646522  27
Mia Han                  7642804  5
Kathleen Herbohn            1736  61
Grace Hsu 

In [28]:
print(rec["fez_record_search_key_issn"])

[{'rek_issn_id': 5855449, 'rek_issn_pid': 'UQ:114d942', 'rek_issn_xsdmf_id': None, 'rek_issn': '1368-3047', 'rek_issn_order': 1, 'fez_sherpa_romeo': {'srm_id': 11124, 'srm_source_id': '2960', 'srm_issn': '1368-3047', 'srm_journal_name': 'Measuring Business Excellence', 'srm_journal_link': 'https://v2.sherpa.ac.uk/id/publication/2960'}, 'fez_ulrichs': {'ulr_issn': '1368-3047', 'ulr_title_id': '265540', 'ulr_title': 'Measuring Business Excellence'}}, {'rek_issn_id': 5855450, 'rek_issn_pid': 'UQ:114d942', 'rek_issn_xsdmf_id': None, 'rek_issn': '1758-8057', 'rek_issn_order': 2, 'fez_sherpa_romeo': {'srm_id': 12042577, 'srm_source_id': '2960', 'srm_issn': '1758-8057', 'srm_journal_name': 'Measuring Business Excellence', 'srm_journal_link': 'https://v2.sherpa.ac.uk/id/publication/2960'}, 'fez_ulrichs': {'ulr_issn': '1758-8057', 'ulr_title_id': '344728', 'ulr_title': 'Measuring Business Excellence'}}]


In [29]:
arts = [x for x in pubs if x["type"] == "Journal Article"]
print(sum(1 for x in arts if x["issns"]), "of", len(arts), "articles have an ISSN")

660 of 664 articles have an ISSN


In [30]:
from collections import Counter
print(Counter(len(x["issns"]) for x in arts))

Counter({2: 479, 1: 173, 3: 8, 0: 4})


In [31]:
for x in arts:
    if not x["issns"]:
        print(f"{x['journal']} | {x['year']} | {x['name']}")

The Practice Manager | 2017 | Robyn King
The Practice Manager | 2015 | Robyn King
The Practice Manager Journal | 2013 | Robyn King
The Practice Manager | 2008 | Robyn King


In [32]:
import pandas as pd
from collections import Counter

abdc = pd.read_excel("ABDC-JQL-2025-v1-260326.xlsx", sheet_name="2025 JQL", header=7)
abdc = abdc.loc[:, ~abdc.columns.astype(str).str.startswith("Unnamed")]
abdc.columns = [str(c).strip() for c in abdc.columns]

lookup = {}
for _, row in abdc.iterrows():
    rating = str(row["2025 rating"]).strip()
    title  = str(row["Journal Title"]).strip()
    for col in ("ISSN", "ISSNOnline"):
        v = str(row[col]).strip()
        if v and v.lower() != "nan":
            lookup[v] = {"rating": rating, "title": title}

for x in pubs:
    hit = next((lookup[i] for i in x.get("issns", []) if i in lookup), None)
    x["abdc"] = hit["rating"] if hit else None
    x["abdc_title"] = hit["title"] if hit else None

arts = [x for x in pubs if x["type"] == "Journal Article"]
print(Counter(x["abdc"] for x in arts))

Counter({'A': 346, 'A*': 161, None: 84, 'B': 52, 'C': 21})


In [33]:
from collections import Counter
print(Counter(x["journal"] for x in arts if not x["abdc"]).most_common())

[('Nature Climate Change', 7), ('SSRN Electronic Journal', 6), ('Bančni vestnik (The Banking Journal)', 4), ('Revizor: revija o reviziji', 4), ('One Earth', 3), ('Biometrics', 3), ('The Practice Manager', 3), ('Computers and Security', 3), ('Nature Communications', 2), ('Scientific Reports', 2), ('World Journal of Surgical Oncology', 2), ('ISACA Journal', 2), ('Advances in Decision Sciences', 1), ('The Journal of Business', 1), ('Materiały i Studia', 1), ('Ethical Investor', 1), ('Corporate Ownership and Control', 1), ('The International Journal of Life Cycle Assessment', 1), ('Climatic Change', 1), ('Accounting and Management Information Systems', 1), ('Humanities and Social Sciences Communications', 1), ('Sankhya: The Indian Journal of Statistics', 1), ('E-Journal of Business Education and Scholarship of Teaching', 1), ('Health Policy and Technology', 1), ('Clinical Imaging', 1), ('Society and Natural Resources', 1), ('CA Magazine', 1), ('Allgemeine Forst- und Jagdzeitung (Journal fo

In [34]:
print(len(arts), "articles")
print(sum(1 for x in arts if x.get("abdc")), "with a rating")
print([x["issns"] for x in arts if x["journal"] and "Banking" in x["journal"]][:5])

664 articles
580 with a rating
[['0378-4266', '1872-6372'], ['0378-4266', '1872-6372'], ['0378-4266', '1872-6372'], ['0378-4266', '1872-6372'], ['0378-4266', '1872-6372']]


In [35]:
sj = pd.read_csv("scimagojr 2025.csv", sep=";")

sj_lookup = {}
for _, row in sj.iterrows():
    issns = [i.strip() for i in str(row["Issn"]).split(",") if i.strip() and i.strip().lower() != "nan"]
    def num(v):
        try:    return float(str(v).replace(",", "."))
        except: return None
    entry = {
        "sjr": num(row["SJR"]),
        "sjr_quartile": str(row["SJR Best Quartile"]).strip(),
        "h_index": row["H index"],
        "cites_per_doc_2y": num(row["Citations / Doc. (2years)"]),
        "sj_title": str(row["Title"]).strip(),
    }
    for i in issns:
        sj_lookup[i] = entry
print(len(sj_lookup), "ISSN entries")

53405 ISSN entries


In [36]:
for x in pubs:
    hit = next((sj_lookup[i.replace("-", "")] for i in x.get("issns", [])
                if i.replace("-", "") in sj_lookup), None)
    x["sjr"]              = hit["sjr"] if hit else None
    x["sjr_quartile"]     = hit["sjr_quartile"] if hit else None
    x["h_index"]          = hit["h_index"] if hit else None
    x["cites_per_doc_2y"] = hit["cites_per_doc_2y"] if hit else None

In [37]:
import time

dois = list({x["doi"] for x in pubs if x.get("doi")})
oa = {}

for i in range(0, len(dois), 50):
    chunk = dois[i:i+50]
    r = requests.get("https://api.openalex.org/works",
                     params={"filter": "doi:" + "|".join(chunk), "per-page": 50},
                     headers={"User-Agent": "UQ-CITS3200 (mailto:you@uwa)"}, timeout=30)
    r.raise_for_status()
    for w in r.json()["results"]:
        key = (w.get("doi") or "").replace("https://doi.org/", "").lower()
        cnp = w.get("citation_normalized_percentile") or {}
        oa[key] = {
            "citation_percentile": cnp.get("value"),
            "cited_by_count": w.get("cited_by_count"),
            "fwci": w.get("fwci"),
        }
    print(f"{i+len(chunk)}/{len(dois)} — {len(oa)} matched")
    time.sleep(1)

for x in pubs:
    hit = oa.get((x.get("doi") or "").lower())
    x["citation_percentile"] = hit["citation_percentile"] if hit else None
    x["cited_by_count"]      = hit["cited_by_count"] if hit else None
    x["fwci"]                = hit["fwci"] if hit else None

arts = [x for x in pubs if x["type"] == "Journal Article"]
print(sum(1 for x in arts if x["citation_percentile"] is not None), "of", len(arts), "articles enriched")

50/609 — 50 matched
100/609 — 100 matched
150/609 — 149 matched
200/609 — 198 matched
250/609 — 247 matched
300/609 — 297 matched
350/609 — 346 matched
400/609 — 395 matched
450/609 — 444 matched
500/609 — 494 matched
550/609 — 543 matched
600/609 — 592 matched
609/609 — 601 matched
605 of 664 articles enriched


In [38]:
vals = [x["citation_percentile"] for x in arts if x["citation_percentile"] is not None]
print(f"min {min(vals):.3f} max {max(vals):.3f} median {sorted(vals)[len(vals)//2]:.3f}")
print(sum(1 for x in arts if x.get("fwci") and x["fwci"] > 2), "papers with FWCI > 2")

min 0.002 max 1.000 median 0.920
374 papers with FWCI > 2


In [39]:
f = sorted(x["fwci"] for x in arts if x.get("fwci") is not None)
print(f"n={len(f)} median={f[len(f)//2]:.2f} p90={f[int(len(f)*0.9)]:.2f} max={f[-1]:.2f}")

n=605 median=2.99 p90=15.57 max=71.14


In [40]:
import statistics
pairs = [(x["citation_percentile"], x["fwci"]) for x in arts
         if x.get("citation_percentile") is not None and x.get("fwci") is not None]
print(len(pairs))
for lo, hi in [(0,0.25),(0.25,0.5),(0.5,0.75),(0.75,1.01)]:
    band = [f for p, f in pairs if lo <= p < hi]
    if band:
        print(f"percentile {lo}-{hi}: n={len(band)} median fwci={statistics.median(band):.2f}")

605
percentile 0-0.25: n=51 median fwci=0.00
percentile 0.25-0.5: n=22 median fwci=0.00
percentile 0.5-0.75: n=42 median fwci=0.43
percentile 0.75-1.01: n=490 median fwci=4.38


In [ ]:
import time, requests

JCR_YEAR = 2025
JCR_BASE = "https://api.clarivate.com/apis/wos-journals/v1"
SLEEP = 0.3                       # API allows 5 req/sec; 2 calls per journal

def num(v):
    try:    return float(v)
    except (TypeError, ValueError): return None

def _get(url, **kw):
    """GET with one retry on rate limit."""
    r = requests.get(url, headers=HDRS, timeout=20, **kw)
    if r.status_code == 429:
        time.sleep(5)
        r = requests.get(url, headers=HDRS, timeout=20, **kw)
    return r

def jcr_lookup(issn):
    r = _get(f"{JCR_BASE}/journals", params={"q": issn, "limit": 5})
    if r.status_code != 200:
        return None
    hits = r.json().get("hits") or []
    if not hits:
        return None
    jid = hits[0]["id"]

    time.sleep(SLEEP)
    r = _get(f"{JCR_BASE}/journals/{jid}/reports/year/{JCR_YEAR}")
    if r.status_code != 200:
        return None
    im = (r.json().get("metrics") or {}).get("impactMetrics") or {}
    return {
        "impact_factor":     num(im.get("jif")),
        "impact_factor_5yr": num(im.get("jif5Years")),
    }

# one lookup per distinct ISSN
issn_set = sorted({i for x in pubs for i in x.get("issns", [])})
cache = {}
for n, issn in enumerate(issn_set, 1):
    try:
        cache[issn] = jcr_lookup(issn)
    except requests.RequestException as e:
        cache[issn] = None
        print(f"  failed {issn}: {e}")
    jif = cache[issn]["impact_factor"] if cache[issn] else None
    print(f"{n}/{len(issn_set)} {issn} -> {jif if jif is not None else '—'}")
    time.sleep(SLEEP)

# attach to pubs
for x in pubs:
    hit = next((cache[i] for i in x.get("issns", []) if cache.get(i)), None)
    x["impact_factor"]     = hit["impact_factor"] if hit else None
    x["impact_factor_5yr"] = hit["impact_factor_5yr"] if hit else None

arts = [x for x in pubs if x["type"] == "Journal Article"]
print(sum(1 for x in arts if x["impact_factor"]), "of", len(arts), "articles have a JIF")

1/353 0001-3072 -> 1.7
2/353 0001-4788 -> 2.5
3/353 0001-4826 -> 4.0
4/353 0002-5852 -> —
5/353 0003-6846 -> 2.6
6/353 0004-9018 -> 0.9
7/353 0004-9158 -> 2.0
8/353 0005-4631 -> —
9/353 0006-341X -> 1.6
10/353 0006-3444 -> 2.8
11/353 0007-6503 -> 3.8
12/353 0007-6813 -> 9.4
13/353 0008-1256 -> 6.1
14/353 0013-0249 -> 0.8
15/353 0017-2723 -> —
16/353 0020-5745 -> —
17/353 0021-9398 -> —
18/353 0022-1082 -> 12.2
19/353 0022-1090 -> 4.1
20/353 0022-2186 -> 1.7
21/353 0024-6301 -> 5.6
22/353 0025-1909 -> 5.7
23/353 0046-3892 -> 7.5
24/353 0047-2506 -> 10.2
25/353 0048-7333 -> 8.9
26/353 0065-0668 -> —
27/353 0095-4918 -> 1.0
28/353 0110-5159 -> —
29/353 0114-0582 -> 2.1
30/353 0140-9883 -> 13.5
31/353 0148-2963 -> 11.1
32/353 0148-558X -> 1.3
33/353 0148-6195 -> 2.8
34/353 0165-0009 -> 5.2
35/353 0165-1765 -> 2.0
36/353 0165-4101 -> 7.8
37/353 0167-2681 -> 2.7
38/353 0167-4048 -> 6.8
39/353 0167-4544 -> 6.3
40/353 0167-7152 -> 0.7
41/353 0167-9473 -> 1.7
42/353 0176-1714 -> 1.3
43/353 0217

In [108]:
print(sum(1 for x in arts if x["impact_factor"]), "of", len(arts))

594 of 662


In [109]:
missing = [(x["journal"], x.get("abdc")) for x in arts
           if not x["impact_factor"] and x.get("abdc") in ("A*", "A")]
from collections import Counter
print(Counter(missing).most_common(15))

[(('Journal of Financial Research', 'A'), 2), (('The Journal of Financial Research', 'A'), 2), (('Advances in Management Accounting', 'A'), 1), (('Advances in Accounting Behavioral Research', 'A'), 1), (('China Accounting and Finance Review', 'A'), 1), (('Australian Tax Forum', 'A*'), 1)]


In [ ]:
r = requests.get(f"{ESPACE_BASE}/journals", headers=HDRS, params={"q": "0002-5852", "limit": 5}, timeout=20)
print(r.status_code, r.json())

200 {'metadata': {'total': 1, 'page': 1, 'limit': 5}, 'hits': [{'id': 'ALLG_FORST_JAGDZTG', 'self': '/journals/ALLG_FORST_JAGDZTG', 'name': 'ALLGEMEINE FORST UND JAGDZEITUNG', 'matches': [{'field': 'issn', 'value': ['<em>0002-5852</em>']}]}]}


In [77]:
for x in pubs:
    if "0002-5852" in x.get("issns", []):
        print(x["name"], "|", x["year"], "|", x["journal"], "|", x["title"][:60], "|", x["link"])

Kathleen Herbohn | 2009 | Allgemeine Forst- und Jagdzeitung (Journal for Forestry and Forest Science ) | Multidimensional performance measurement systems – A promisi | https://espace.library.uq.edu.au/view/UQ:197574


In [78]:
import json, pandas as pd
from datetime import datetime, timezone

# 1. staff (was people)
staff = [{
    "name": p["name_clean"],
    "job_title": p["title_clean"],
    "academic_level": LEVEL.get(p["title_clean"]),
    "university": p["university"],
    "field_of_research": p["discipline"],
    "espace_id": p.get("espace_id"),
    "profile_url": p["profile_url"],
} for p in records]

# 2. journals
journals = {}
for x in pubs:
    if not x["journal"]:
        continue
    key = x.get("abdc_title") or x["journal"] or "unknown"
    if key not in journals:
        journals[key] = {
            "journal_name": key,
            "journal": x["journal"],
            "journal_canonical": x.get("journal_canonical"),
            "publisher": x.get("publisher"),
            "issn": "; ".join(x["issns"]) if x["issns"] else None,
            "quality_rank": x.get("abdc"),
            "impact_factor": x.get("impact_factor"),
            "impact_factor_5yr": x.get("impact_factor_5yr"),
            "sjr": x.get("sjr"),
            "sjr_quartile": x.get("sjr_quartile"),
            "h_index": x.get("h_index"),
            "cites_per_doc_2y": x.get("cites_per_doc_2y"),
        }
journals = list(journals.values())

# 3. publications
_seen = set()
publications = []
for x in sorted(pubs, key=lambda r: (r.get("doi") is None)):
    if x["type"] != "Journal Article":
        continue
    k = (x["name"], x["title"].lower().strip(), x["year"])
    if k in _seen:
        continue
    _seen.add(k)
    publications.append({
        "espace_id": x["espace_id"],
        "name": x["name"],
        "journal_name": x.get("abdc_title") or x["journal"] or "unknown",
        "title": x["title"],
        "year": x["year"],
        "author_count": x["n_authors"],
        "authors": x["authors"],
        "quality_rank": x.get("abdc"),
        "sjr_quartile": x.get("sjr_quartile"),
        "doi": x.get("doi"),
        "article_url": f"https://doi.org/{x['doi']}" if x.get("doi") else None,
        "link": x["link"],
        "source": "UQ eSpace",
        "citation_percentile": x.get("citation_percentile"),
        "cited_by_count": x.get("cited_by_count"),
        "fwci": x.get("fwci"),
    })


harvest = [{
    "university": "University of Queensland",
    "source": "UQ eSpace",
    "last_run": datetime.now(timezone.utc).isoformat(),
    "latest_year": max((int(p["year"]) for p in publications if p["year"]), default=None),
}]

for name, data in [("staff", staff), ("journals", journals), ("publications", publications), ("harvest", harvest)]:
    with open(f"{name}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    pd.DataFrame(data).to_csv(f"{name}.csv", index=False)
    print(name, len(data))



staff 43
journals 207
publications 662
harvest 1


In [79]:
print(len(journals), "journals;", len({j["journal_name"] for j in journals}), "distinct names")

207 journals; 207 distinct names


In [80]:
from collections import Counter
dupes = [n for n, c in Counter(j["journal"] for j in journals).items() if c > 1]
for j in journals:
    if j["journal"] in dupes:
        print(f"{j['journal']:45} {j['journal_key']:12} {j['abdc']}")

In [81]:
print(len({x.get("abdc_title") or x["journal"] for x in pubs if x["journal"]}), "distinct journals in pubs")
print(len(journals), "rows in journals table")

207 distinct journals in pubs
207 rows in journals table


In [82]:
from collections import Counter
print([n for n, c in Counter(p["name_clean"] for p in records).items() if c > 1])   # joint appointments
print([p["name_clean"] for p in records if not p["title_clean"]])                   # ranks LADDER missed

[]
['Chris Bell']


In [83]:
from collections import Counter
arts = [x for x in pubs if x["type"] == "Journal Article"]

_s, _d = set(), []
for x in sorted(arts, key=lambda r: (r.get("doi") is None)):
    k = (x["name"], x["title"].lower().strip(), x["year"])
    if k not in _s:
        _s.add(k)
        _d.append(x)
arts = _d
print(Counter(x["n_authors"] for x in arts).most_common(10))

[(3, 255), (4, 179), (2, 130), (5, 43), (1, 38), (6, 6), (10, 3), (13, 2), (80, 1), (16, 1)]


In [84]:
print([(x["title"][:60], x["n_authors"], x["journal"]) for x in arts if x["n_authors"] > 15])

[('Nonstandard errors', 80, 'The Journal of Finance'), ('Fantasy pitching', 16, 'Journal of Accounting and Management Information Systems'), ('Motivating postgrad research students to pitch their ideas: ', 25, 'SSRN Electronic Journal')]


In [85]:
print([k for k in rec if "doi" in k.lower()])

['fez_record_search_key_doi', 'fez_record_search_key_new_doi']


In [86]:
print(rec.get("fez_record_search_key_doi"))
print(rec.get("fez_record_search_key_new_doi"))

{'rek_doi_id': 1911839, 'rek_doi_pid': 'UQ:114d942', 'rek_doi_xsdmf_id': None, 'rek_doi': '10.1108/MBE-12-2015-0055', 'fez_altmetric': {'as_id': 266071, 'as_amid': 18658456, 'as_doi': '10.1108/MBE-12-2015-0055', 'as_score': 1, 'as_created': '2022-12-23 09:00:56', 'as_last_checked': '2026-08-09 08:10:15', 'as_1d': 0, 'as_2d': 0, 'as_3d': 0, 'as_4d': 0, 'as_5d': 0, 'as_6d': 0, 'as_1w': 0, 'as_1m': 0, 'as_3m': 0, 'as_6m': 0, 'as_1y': 0, 'as_total_posts_count': 1, 'as_facebook_posts_count': 0, 'as_policy_posts_count': 0, 'as_blogs_posts_count': 0, 'as_googleplus_posts_count': 0, 'as_news_posts_count': 0, 'as_reddit_posts_count': 0, 'as_twitter_posts_count': 0, 'as_syllabi_posts_count': 0, 'as_video_posts_count': 0, 'as_weibo_posts_count': 0, 'as_qa_posts_count': 0, 'as_f1000_posts_count': 0, 'as_wikipedia_posts_count': 0, 'as_pinterest_posts_count': 0, 'as_linkedin_posts_count': 0, 'as_peer_reviews_posts_count': 1, 'as_bluesky_posts_count': 0, 'as_guideline_posts_count': 0, 'as_patent_post

In [87]:
print(arts[0]["doi"])

10.1142/s1094406025410057


In [88]:
from collections import Counter
dois = [x["doi"] for x in arts if x.get("doi")]
print(len(dois), "of", len(arts), "have a DOI")
print([d for d, c in Counter(dois).items() if c > 1][:10])

616 of 662 have a DOI
['10.1142/s1094406025410057', '10.1016/j.jcae.2026.100571', '10.1108/jal-02-2025-0074', '10.1111/jfir.70021', '10.1016/j.gfj.2025.101173', '10.1111/jfir.70010', '10.1111/corg.12623', '10.1016/j.jfs.2025.101384', '10.1080/09638180.2024.2368129', '10.1016/j.pacfin.2023.102008']


In [89]:
from collections import Counter
c = Counter(dois)
dupes = {d: n for d, n in c.items() if n > 1}
print(len(dupes), "papers appear more than once")
print(Counter(dupes.values()))          # how many appear 2x, 3x, ...
print(len(arts) - sum(n - 1 for n in dupes.values()), "unique papers")

51 papers appear more than once
Counter({2: 48, 3: 2, 4: 1})
607 unique papers


In [90]:
from collections import Counter
nodoi = [x for x in arts if not x.get("doi")]
print(Counter(x["year"] for x in nodoi).most_common(10))
print(Counter(x["journal"] for x in nodoi).most_common(10))

[('2011', 4), ('2008', 4), ('2016', 3), ('2010', 3), ('2009', 3), ('2007', 3), ('2018', 2), ('2025', 2), ('2022', 2), ('2021', 2)]
[('IMA Educational Case Journal', 4), ('Bančni vestnik (The Banking Journal)', 4), ('Revizor: revija o reviziji', 4), ('The Practice Manager', 3), ('Economic and Business Review: for Central and South-Eastern Europe', 3), ('ISACA Journal', 2), ('Advances in Decision Sciences', 1), ('Ethical Investor', 1), ('Accounting and Management Information Systems', 1), ('Sankhya: The Indian Journal of Statistics', 1)]


In [91]:
for x in nodoi:
    if x["year"] in ("2024", "2025", "2026"):
        print(x["year"], "|", x["journal"], "|", x["title"][:60], "|", x.get("abdc"))

2025 | Australian Tax Forum | Social norms and tax compliance intention during COVID-19: t | A*
2025 | ISACA journal | Optimizing supplier cyberrisk assessment | None
2024 | Internal Auditor | Cybersecurity assurance | None
2024 | ISACA Journal | The three lines model in cybersecurity governance and risk m | None


In [92]:
for x in arts:
    if x["journal"] == "Australian Tax Forum" :
        print(x["year"], x.get("doi"), x["link"])

2025 None https://espace.library.uq.edu.au/view/UQ:66e610d


In [93]:
import requests, urllib.parse

def find_doi(title, journal=None, year=None):
    params = {"query.bibliographic": title, "rows": 3}
    if journal:
        params["query.container-title"] = journal
    r = requests.get("https://api.crossref.org/works",
                     params=params,
                     headers={"User-Agent": "UQ-CITS3200/1.0 (mailto:24314165@student.uwa.edu.au)"},
                     timeout=15)
    for item in r.json()["message"]["items"]:
        t = (item.get("title") or [""])[0]
        y = item.get("issued", {}).get("date-parts", [[None]])[0][0]
        if t.lower().strip() == title.lower().strip() and (not year or str(y) == str(year)):
            return item["DOI"]
    return None

In [94]:
for x in arts:
    if x["journal"] == "Australian Tax Forum" and not x.get("doi"):
        print(find_doi(x["title"], x["journal"], x["year"]), "|", x["title"][:60])

None | Social norms and tax compliance intention during COVID-19: t


In [95]:
x = next(x for x in arts if x["journal"] == "Australian Tax Forum" and not x.get("doi"))
print("OURS:", repr(x["title"]), x["year"])

r = requests.get("https://api.crossref.org/works",
                 params={"query.bibliographic": x["title"], "rows": 5},
                 headers={"User-Agent": "UQ-CITS3200/1.0"}, timeout=15)
for item in r.json()["message"]["items"]:
    print(item["DOI"], "|", item.get("issued",{}).get("date-parts",[[None]])[0][0],
          "|", (item.get("title") or [""])[0][:70],
          "|", (item.get("container-title") or [""])[0][:40])

OURS: 'Social norms and tax compliance intention during COVID-19: the case of JobKeeper' 2025
10.36262/widyakala.v10i1.711 | 2023 | The Impact of Tax Reform, Subjective Norms and Social Norms on SME’s T | WIDYAKALA JOURNAL : JOURNAL OF PEMBANGUN
10.7176/rjfa/13-1-04 | 2022 | THE EFFECT OF TAX INCENTIVES AND TAX KNOWLEDGE ON CORPORATE TAXPAYER C | Research Journal of Finance and Accounti
10.21203/rs.3.rs-2075093/v1 | 2022 | Do Operational Facilitating Conditions and Subjective Norms Improve th | 
10.18502/kss.v7i14.11980 | 2022 | MSME Tax Compliance During the COVID-19 Pandemic | KnE Social Sciences
10.1109/cnn63506.2024.10705834 | 2024 | Social Norm Compliance for Newly Introduced Norms During COVID-19 Pand | 2024 Sixth International Conference Neur


In [96]:
r = requests.get("https://api.crossref.org/works/10.3316/informit.T2025091300002590064381796",
                 headers={"User-Agent": "UQ-CITS3200/1.0"}, timeout=15)
print(r.status_code)

404


In [97]:
r = requests.get("https://api.openalex.org/works",
                 params={"search": x["title"], "per_page": 5},
                 headers={"User-Agent": "UQ-CITS3200 (mailto:you@uq)"}, timeout=15)
for w in r.json()["results"]:
    print(w.get("doi"), "|", w.get("publication_year"), "|", w["display_name"][:60])

https://doi.org/10.13140/rg.2.2.24794.88004 | 2021 | COVID-19: Rental housing and homelessness impacts - an initi
None | 2020 | Community group buying, vulnerable communities and COVID-19 
https://doi.org/10.32613/nr/2022.3 | 2022 | Really proper dangerous one: Aboriginal responses to the fir
https://doi.org/10.1007/978-3-030-97258-5 | 2023 | The Forum of Federations Handbook of Fiscal Federalism
https://doi.org/10.4324/9781003196020-11 | 2022 | Digitally Prepared?


In [98]:
r = requests.get("https://api.datacite.org/dois",
                 params={"query": x["title"], "page[size]": 5}, timeout=15)

In [99]:
for d in r.json()["data"][:5]:
    a = d["attributes"]
    print(a["doi"], "|", a.get("publicationYear"), "|", (a.get("titles") or [{}])[0].get("title", "")[:60])

In [100]:
j = r.json()
print(r.status_code, len(j.get("data", [])), j.get("meta", {}).get("total"))

200 0 0


In [101]:
sj = pd.read_csv("scimagojr 2025.csv", sep=";")
print(len(sj), "journals")
print(sj.columns.tolist())
print(sj[["Title", "Issn", "SJR", "SJR Best Quartile", "H index"]].head())

32193 journals
['Rank', 'Sourceid', 'Title', 'Type', 'Issn', 'Publisher', 'Open Access', 'Open Access Diamond', 'SJR', 'SJR Best Quartile', 'H index', 'Total Docs. (2025)', 'Total Docs. (3years)', 'Total Refs.', 'Total Citations (3years)', 'Citable Docs. (3years)', 'Citations / Doc. (2years)', 'Ref. / Doc.', '%Female', 'Overton', 'Country', 'Region', 'Publisher.1', 'Coverage', 'Categories', 'Areas']
                                   Title                Issn      SJR  \
0     Ca-A Cancer Journal for Clinicians  15424863, 00079235  104,065   
1  Nature Reviews Molecular Cell Biology  14710072, 14710080   35,568   
2         Quarterly Journal of Economics  00335533, 15314650   33,069   
3          Nature Reviews Drug Discovery  14741784, 14741776   32,577   
4       Nature Reviews Clinical Oncology  17594782, 17594774   28,076   

  SJR Best Quartile  H index  
0                Q1      236  
1                Q1      553  
2                Q1      336  
3                Q1      428  
4  

In [102]:
print([(p["name_clean"], p["title_clean"]) for p in records if not LEVEL.get(p["title_clean"])])

[('Chris Bell', None), ('Yong Ming Chen', 'Teaching Associate')]


In [103]:
import difflib
for i, a in enumerate(arts):
    for b in arts[i+1:]:
        if a["name"] != b["name"] or not a.get("title") or not b.get("title"):
            continue
        r = difflib.SequenceMatcher(None, a["title"].lower(), b["title"].lower()).ratio()
        if r > 0.95 and a["title"] != b["title"]:
            print(f"{r:.2f} {a['year']} {a['title'][:50]} | {a.get('doi')}")
            print(f"     {b['year']} {b['title'][:50]} | {b.get('doi')}\n")

In [104]:
for x in arts:
    if x.get("title") and "determinants of credit ratings" in x["title"].lower():
        print(x["title"][:55])
        print("   journal:", x["journal"])
        print("   issns:", x["issns"], "| abdc:", x.get("abdc"))
        print("   doi:", x.get("doi"), "| link:", x["link"], "\n")

The Determinants of Credit Ratings: Australian Evidence
   journal: Australian Journal of Management
   issns: ['0312-8962'] | abdc: A
   doi: 10.1177/031289620603100208 | link: https://espace.library.uq.edu.au/view/UQ:712863 



In [105]:
d = next(p["doi"] for p in publications if p.get("doi"))
r = requests.get(f"https://api.openalex.org/works/doi:{d}",
                 headers={"User-Agent": "UQ-CITS3200 (mailto:you@uwa)"}, timeout=15)
w = r.json()
print(r.status_code)
print(w.get("citation_normalized_percentile"))
print(w.get("cited_by_count"), w.get("fwci"))

200
{'value': 0.20402102, 'is_in_top_1_percent': False, 'is_in_top_10_percent': False}
0 0.0


In [106]:
import json, requests
import os
from dotenv import load_dotenv

load_dotenv()
KEY = os.environ["CLARIVATE_API_KEY"]                    
HDRS = {"X-ApiKey": KEY}

r = requests.get("https://api.clarivate.com/apis/wos-journals/v1/journals/J_CORP_ACCOUNT_FINAN",
                 headers=HDRS, timeout=15)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:4000])

200
{
  "id": "J_CORP_ACCOUNT_FINAN",
  "name": "Journal of Corporate Accounting and Finance",
  "jcrTitle": "J CORP ACCOUNT FINAN",
  "isoTitle": "J. Corp. Account. Financ.",
  "issn": "1044-8136",
  "previousIssn": [],
  "eIssn": "1097-0053",
  "publisher": {
    "name": "WILEY",
    "address": "111 RIVER ST, HOBOKEN 07030-5774, NJ",
    "countryRegion": "USA"
  },
  "frequency": 6,
  "firstIssueYear": 1989,
  "language": "English",
  "categories": [
    {
      "url": "/categories/DK_ESCI",
      "name": "BUSINESS, FINANCE",
      "edition": "ESCI"
    }
  ],
  "journalCitationReports": [
    {
      "year": 2025,
      "url": "/journals/J_CORP_ACCOUNT_FINAN/reports/year/2025"
    },
    {
      "year": 2024,
      "url": "/journals/J_CORP_ACCOUNT_FINAN/reports/year/2024"
    },
    {
      "year": 2023,
      "url": "/journals/J_CORP_ACCOUNT_FINAN/reports/year/2023"
    },
    {
      "year": 2022,
      "url": "/journals/J_CORP_ACCOUNT_FINAN/reports/year/2022"
    },
    {
      "

In [107]:
r = requests.get("https://api.clarivate.com/apis/wos-journals/v1/journals/J_CORP_ACCOUNT_FINAN/reports/year/2025",
                 headers=HDRS, timeout=15)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:4000])

200
{
  "year": 2025,
  "suppressed": false,
  "onHold": false,
  "delisted": false,
  "journal": {
    "id": "J_CORP_ACCOUNT_FINAN",
    "self": "/journals/J_CORP_ACCOUNT_FINAN",
    "name": "Journal of Corporate Accounting and Finance"
  },
  "metrics": {
    "impactMetrics": {
      "totalCites": 688,
      "jif": "1.3",
      "jifWithoutSelfCitations": "1.3",
      "jif5Years": 1.6,
      "immediacyIndex": 0.2,
      "jci": 0.3
    },
    "influenceMetrics": {
      "eigenFactor": {
        "score": 0.00052,
        "normalized": 0.11834
      },
      "articleInfluence": 0.207
    },
    "sourceMetrics": {
      "citableItems": {
        "total": 52,
        "articlesPercentage": 100.0
      },
      "jifPercentile": 34.8,
      "halfLife": {
        "cited": 4.4,
        "citing": 13.7
      }
    },
    "citationDistribution": {
      "articleCitationMedian": 1.0,
      "reviewCitationMedian": 1.0,
      "unlinkedCitations": 6,
      "timesCited": 0,
      "articles": 62,
      